In [1]:
# MobileNetV3 Large model load

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import json
import os
import faiss
import time


# MobileNetV3 Large 모델 로드


def load_mobilenet_v3():
    global model, preprocess, EMB_SIZE

    print("MobileNetV3 Large 모델 로드 중...")

    weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
    model = models.mobilenet_v3_large(weights=weights)

    # 최종 FC 제거하여 임베딩 출력
    model.classifier = nn.Identity()
    EMB_SIZE = 960  # MobileNetV3 Large 출력 차원

    preprocess = weights.transforms()  # 자동 Resize/Normalize

    model.eval()

    # ---- warm-up ----
    dummy = torch.zeros(1, 3, 224, 224)
    with torch.no_grad():
        model(dummy)

    print("MobileNetV3 Large Warm-up 완료!")
    print(f"임베딩 차원: {EMB_SIZE}\n")

load_mobilenet_v3()

🚀 MobileNetV3 Large 모델 로드 중...
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /Users/a/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:05<00:00, 4.41MB/s]


🔥 MobileNetV3 Large Warm-up 완료!
임베딩 차원: 960



In [2]:
# 이미지 다운로드 + 전처리 함수

def load_and_preprocess(url):
    try:
        response = requests.get(url, timeout=(2, 5))
        response.raise_for_status()

        img = Image.open(BytesIO(response.content)).convert("RGB")
        img_tensor = preprocess(img).unsqueeze(0)

        return img_tensor

    except Exception as e:
        print(f"이미지 로드 실패: {url}")
        print("사유:", e)
        return None

In [3]:
# 임베딩 생성

def get_embedding(url):
    img_tensor = load_and_preprocess(url)
    if img_tensor is None:
        return None

    with torch.no_grad():
        emb = model(img_tensor).squeeze().numpy().astype("float32")

    # L2 정규화
    emb = emb / (np.linalg.norm(emb) + 1e-10)

    return emb

In [6]:
# ssadagu Json 불러오기 , 상품 메타데이터 파싱

JSON_PATH = "/Users/a/IdeaProjects/Final-AI/dev/crawling_tests/ssadagu_search_results.json"

with open(JSON_PATH, "r") as f:
    data = json.load(f)

products = data["products"]

print(f"총 상품 수: {len(products)}개")

총 상품 수: 30개


In [7]:
# 모든 상품 임베딩 생성 -> emb_matrix & id_map 만들기

emb_list = []
id_map = {}

print("모든 상품 이미지 → MobileNet 임베딩 생성 시작\n")

for idx, item in enumerate(products):
    url = item["thumbnail_url"]

    emb = get_embedding(url)
    if emb is None:
        continue

    emb_list.append(emb)
    id_map[idx] = {
        "index": idx,
        "title": item["title"],
        "price": item["price"],
        "product_link": item["product_link"],
        "thumbnail_url": item["thumbnail_url"]
    }

    print(f"[{idx}] 임베딩 생성 완료")

print("\n 전체 임베딩 생성 완료!")

🔥 모든 상품 이미지 → MobileNet 임베딩 생성 시작

[0] 임베딩 생성 완료
[1] 임베딩 생성 완료
[2] 임베딩 생성 완료
[3] 임베딩 생성 완료
[4] 임베딩 생성 완료
[5] 임베딩 생성 완료
[6] 임베딩 생성 완료
[7] 임베딩 생성 완료
[8] 임베딩 생성 완료
[9] 임베딩 생성 완료
[10] 임베딩 생성 완료
[11] 임베딩 생성 완료
[12] 임베딩 생성 완료
[13] 임베딩 생성 완료
[14] 임베딩 생성 완료
[15] 임베딩 생성 완료
[16] 임베딩 생성 완료
[17] 임베딩 생성 완료
[18] 임베딩 생성 완료
[19] 임베딩 생성 완료
[20] 임베딩 생성 완료
[21] 임베딩 생성 완료
[22] 임베딩 생성 완료
[23] 임베딩 생성 완료
[24] 임베딩 생성 완료
[25] 임베딩 생성 완료
[26] 임베딩 생성 완료
[27] 임베딩 생성 완료
[28] 임베딩 생성 완료
[29] 임베딩 생성 완료

🔥 전체 임베딩 생성 완료!


In [8]:
# emb_matrix 저장

emb_matrix = np.vstack(emb_list).astype("float32")

np.save("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/emb_matrix_mobilenet.npy", emb_matrix)

print("emb_matrix 저장 완료!\n")

💾 emb_matrix 저장 완료!



In [9]:
# id_map.json 저장

with open("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/id_map_mobilenet.json", "w") as f:
    json.dump(id_map, f, indent=2)

print("id_map 저장 완료!\n")

💾 id_map 저장 완료!



In [10]:
# FAISS index 저장

dim = EMB_SIZE
index = faiss.IndexFlatL2(dim)

# FAISS 정규화 필수
faiss.normalize_L2(emb_matrix)

index.add(emb_matrix)

faiss.write_index(index, "/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/faiss_index_mobilenet.bin")

print("FAISS index 저장 완료!")

💾 FAISS index 저장 완료!


In [11]:
# 검색 함수

# 로드
emb_matrix = np.load("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/emb_matrix_mobilenet.npy")
index = faiss.read_index("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/faiss_index_mobilenet.bin")

with open("/Users/a/IdeaProjects/Final-AI/dev/cnn_test/mobilenet_file/id_map_mobilenet.json", "r") as f:
    id_map = json.load(f)

# 정규화
faiss.normalize_L2(emb_matrix)


def search_similar(url, top_k=5):
    print("\n search_similar 실행!")

    t0 = time.time()

    emb = get_embedding(url)
    if emb is None:
        return []

    vec = emb.reshape(1, -1)

    D, I = index.search(vec, top_k)

    results = [id_map[str(i)] for i in I[0]]

    print(f" 실행시간: {round(time.time() - t0, 4)}초")

    return results

In [ ]:
# 테스트 제발 되라.....

test_url = "https://thumbnail.coupangcdn.com/thumbnails/remote/320x320ex/image/retail/images/6671171604579-e3dec662-3729-4133-8fd9-107e14004798.jpg"

results = search_similar(test_url, top_k=5)

print("\n=== 검색 결과 ===")
for r in results:
    print(f"- {r['title']} | ₩{r['price']} | {r['product_link']}")


🔥 search_similar 실행!
